In [1]:
import grid2op
from grid2op.Environment import Environment
from registry_object import EnvironmentManager
import numpy as np
import logging
import sys

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

# 创建环境管理器
env_manager = EnvironmentManager()

# 选择一个Grid2Op环境并注册
env_name = "l2rpn_case14_sandbox"  # 或选择其他可用环境
env_id = "test_env"

# 创建环境
env = grid2op.make(env_name)
env_manager.register_env(env_id, env)

# 获取一个初始观察
obs = env.reset()
env_manager.record_observation(env_id, obs)

2025-03-05 15:02:41,800 - pandapower.convert_format - INFO - These dtypes could not be corrected: {'trafo': ['tap_min', 'tap_max']}


In [2]:
def test_observation_function(func, env_manager, env_id, **kwargs):
    """
    测试一个观察函数并打印结果
    
    参数:
    func: 要测试的函数
    env_manager: 环境管理器实例
    env_id: 环境ID
    **kwargs: 传递给函数的其他参数
    """
    func_name = func.__name__
    print(f"\n{'='*50}")
    print(f"测试函数: {func_name}")
    
    try:
        result = func(env_manager, env_id, **kwargs)
        print(f"状态: {result['status']}")
        print(f"消息: {result['message']}")
        
        if result['status'] == 'success' and 'data' in result:
            if isinstance(result['data'], (list, np.ndarray)) and len(result['data']) > 5:
                # 如果数据量大，只打印部分
                print(f"数据 (前5项): {result['data'][:5]}")
                print(f"数据长度: {len(result['data'])}")
            else:
                print(f"数据: {result['data']}")
        
        return True, result
    except Exception as e:
        print(f"异常: {str(e)}")
        return False, None

In [3]:
from execution_func import *

def test_all_observation_functions(env_manager, env_id):
    """测试所有观察函数"""
    results = {}
    
    # 测试发电机相关观测函数
    print("\n\n*** 测试发电机相关观测函数 ***")
    gen_funcs = [
        obs_gen_p_impl, obs_gen_q_impl, obs_gen_v_impl, obs_gen_theta_impl,
        obs_target_dispatch_impl, obs_actual_dispatch_impl, 
        obs_gen_p_before_curtail_impl, obs_curtailment_mw_impl,
        obs_curtailment_impl, obs_curtailment_limit_impl,
        obs_gen_margin_up_impl, obs_gen_margin_down_impl
    ]
    
    for func in gen_funcs:
        # 先测试获取所有发电机数据
        success, result = test_observation_function(func, env_manager, env_id)
        results[func.__name__ + "_all"] = success
        
        if success and len(result.get('data', [])) > 0:
            # 然后测试获取特定发电机数据
            gen_id = [0]  # 测试第一个发电机
            success, _ = test_observation_function(func, env_manager, env_id, gen_id=gen_id)
            results[func.__name__ + "_specific"] = success
    
    # 测试负载相关观测函数
    print("\n\n*** 测试负载相关观测函数 ***")
    load_funcs = [
        obs_load_p_impl, obs_load_q_impl, obs_load_v_impl, obs_load_theta_impl
    ]
    
    for func in load_funcs:
        success, result = test_observation_function(func, env_manager, env_id)
        results[func.__name__ + "_all"] = success
        
        if success and len(result.get('data', [])) > 0:
            load_id = [0]  # 测试第一个负载
            success, _ = test_observation_function(func, env_manager, env_id, load_id=load_id)
            results[func.__name__ + "_specific"] = success
    
    # 测试线路相关观测函数
    print("\n\n*** 测试线路相关观测函数 ***")
    line_funcs = [
        obs_p_or_impl, obs_q_or_impl, obs_v_or_impl, obs_a_or_impl, obs_theta_or_impl,
        obs_p_ex_impl, obs_q_ex_impl, obs_v_ex_impl, obs_a_ex_impl, obs_theta_ex_impl,
        obs_rho_impl, obs_line_status_impl, obs_timestep_overflow_impl,
        obs_time_before_cooldown_line_impl, obs_time_next_maintenance_impl,
        obs_duration_next_maintenance_impl
    ]
    
    for func in line_funcs:
        success, result = test_observation_function(func, env_manager, env_id)
        results[func.__name__ + "_all"] = success
        
        if success and len(result.get('data', [])) > 0:
            line_id = [0]  # 测试第一条线路
            success, _ = test_observation_function(func, env_manager, env_id, line_id=line_id)
            results[func.__name__ + "_specific"] = success
    
    # 测试变电站相关观测函数
    print("\n\n*** 测试变电站相关观测函数 ***")
    success, result = test_observation_function(obs_time_before_cooldown_sub_impl, env_manager, env_id)
    results["obs_time_before_cooldown_sub_impl_all"] = success
    
    if success and len(result.get('data', [])) > 0:
        sub_id = [0]  # 测试第一个变电站
        success, _ = test_observation_function(obs_time_before_cooldown_sub_impl, env_manager, env_id, sub_id=sub_id)
        results["obs_time_before_cooldown_sub_impl_specific"] = success
    
    # 测试储能设备相关观测函数
    print("\n\n*** 测试储能设备相关观测函数 ***")
    storage_funcs = [
        obs_storage_charge_impl, obs_storage_power_target_impl,
        obs_storage_power_impl, obs_storage_theta_impl
    ]
    
    for func in storage_funcs:
        success, result = test_observation_function(func, env_manager, env_id)
        results[func.__name__ + "_all"] = success
        
        if success and len(result.get('data', [])) > 0:
            storage_id = [0]  # 测试第一个储能设备
            success, _ = test_observation_function(func, env_manager, env_id, storage_id=storage_id)
            results[func.__name__ + "_specific"] = success
    
    # 测试系统整体观测函数
    print("\n\n*** 测试系统整体观测函数 ***")
    system_funcs = [
        obs_date_time_impl, obs_topo_vect_impl, obs_max_step_impl,
        obs_current_step_impl, obs_delta_time_impl
    ]
    
    for func in system_funcs:
        success, _ = test_observation_function(func, env_manager, env_id)
        results[func.__name__] = success
    
    # 测试告警系统相关观测函数
    print("\n\n*** 测试告警系统相关观测函数 ***")
    alert_funcs = [
        obs_is_alarm_illegal_impl, obs_time_since_last_alarm_impl,
        obs_last_alarm_impl, obs_attention_budget_impl,
        obs_total_number_of_alert_impl, obs_was_alert_used_after_attack_impl,
        obs_attack_under_alert_impl, obs_time_since_last_alert_impl,
        obs_alert_duration_impl, obs_time_since_last_attack_impl
    ]
    
    for func in alert_funcs:
        success, _ = test_observation_function(func, env_manager, env_id)
        results[func.__name__] = success
    
    # 打印汇总结果
    print("\n\n*** 测试结果汇总 ***")
    success_count = sum(1 for result in results.values() if result)
    total_count = len(results)
    print(f"成功测试: {success_count}/{total_count} ({success_count/total_count*100:.2f}%)")
    
    # 列出失败的测试
    if success_count < total_count:
        print("\n失败的测试:")
        for name, success in results.items():
            if not success:
                print(f"- {name}")
    
    return results

# 执行所有测试
test_results = test_all_observation_functions(env_manager, env_id)

2025-03-05 15:02:45,182 - src.utils - INFO - 这是带时间戳的日志文件测试信息。


*** 测试发电机相关观测函数 ***

测试函数: obs_gen_p_impl
2025-03-05 15:02:45,185 - execution_func - INFO - 开始获取所有发电机的有功功率，env_id: test_env
2025-03-05 15:02:45,186 - execution_func - INFO - 成功获取所有发电机的有功功率。
状态: success
消息: 成功获取所有发电机的有功功率。
数据 (前5项): [81.4000015258789, 79.30000305175781, 5.300000190734863, 0.0, 0.0]
数据长度: 6

测试函数: obs_gen_p_impl
2025-03-05 15:02:45,187 - execution_func - INFO - 开始获取指定发电机的有功功率，env_id: test_env, gen_id: [0]
2025-03-05 15:02:45,188 - execution_func - INFO - 成功获取指定发电机的有功功率。
状态: success
消息: 成功获取指定发电机的有功功率。
数据: [81.4000015258789]

测试函数: obs_gen_q_impl
2025-03-05 15:02:45,190 - execution_func - INFO - 开始获取所有发电机的无功功率，env_id: test_env
2025-03-05 15:02:45,190 - execution_func - INFO - 成功获取所有发电机的无功功率。
状态: success
消息: 成功获取所有发电机的无功功率。
数据 (前5项): [19.49603843688965, 71.3402328491211, 24.36892318725586, 24.36892318725586, 24.018070220947266]
数据长度: 6

测试函数: obs_gen_q_impl
2025-03-05 15:02:45,191 - execution_func - INFO - 开始获

In [4]:
def test_advanced_features(env_manager, env_id):
    """测试需要特定环境配置的功能"""
    
    # 1. 测试重调度相关功能
    print("\n\n*** 测试重调度场景 ***")
    env = env_manager._envs.get(env_id)
    
    # 执行重调度操作
    gen_id = [0]  # 选择一个可调节的发电机
    amount = [5.0]  # 增加5MW的发电量
    redispatch_result = redispatch_impl(env_manager, env_id, gen_id, amount)
    print(f"重调度操作结果: {redispatch_result['status']}")
    
    # 检查目标调度和实际调度
    target_dispatch = test_observation_function(obs_target_dispatch_impl, env_manager, env_id, gen_id=gen_id)[1]
    actual_dispatch = test_observation_function(obs_actual_dispatch_impl, env_manager, env_id, gen_id=gen_id)[1]
    print(f"目标调度值: {target_dispatch['data']}")
    print(f"实际调度值: {actual_dispatch['data']}")
    
    # 2. 测试告警系统功能 (如果环境支持)
    print("\n\n*** 测试告警系统功能 ***")
    try:
        # 检查是否有告警功能
        alert_budget = test_observation_function(obs_attention_budget_impl, env_manager, env_id)[1]
        if alert_budget['status'] == 'success':
            print(f"当前注意力预算: {alert_budget['data']}")
            
            # 检查告警历史
            alerts = test_observation_function(obs_total_number_of_alert_impl, env_manager, env_id)[1]
            print(f"已发出的告警总数: {alerts['data']}")
        else:
            print("当前环境不支持告警功能")
    except Exception as e:
        print(f"测试告警系统时出错: {str(e)}")

In [5]:
def check_attribute_exists(env_manager, env_id, attribute_name):
    """检查观察对象是否有特定属性"""
    obs = env_manager.get_latest_obs(env_id)
    return hasattr(obs, attribute_name)

# 使用示例
for attr in ['curtailment', 'storage_charge', 'attention_budget']:
    exists = check_attribute_exists(env_manager, env_id, attr)
    print(f"属性 '{attr}' {'存在' if exists else '不存在'}")

属性 'curtailment' 存在
属性 'storage_charge' 存在
属性 'attention_budget' 存在


In [6]:
def check_env_features(env_manager, env_id):
    """检查环境支持的功能"""
    env = env_manager._envs.get(env_id)
    features = {
        "redispatching": hasattr(env, "redispatching_unit_commitment_availble"),
        "storage": hasattr(env, "storage_units_available"),
        "alarm": hasattr(env, "alarms_area_names")
    }
    return features

In [7]:
# 例如，要测试重调度观察，先执行重调度操作
redispatch_impl(env_manager, env_id, [0], [5.0])
# 然后测试相关观察
test_observation_function(obs_target_dispatch_impl, env_manager, env_id)

2025-03-05 15:02:45,360 - execution_func - INFO - 开始执行 redispatch 操作，env_id: test_env, gen_id: [0], amount: [5.0]
This action will:
	 - NOT change anything to the injections
	 - Modify the generators with redispatching in the following way:
	 	 - Redispatch "gen_1_0" of 5.00 MW
	 - NOT modify any storage capacity
	 - NOT perform any curtailment
	 - NOT force any line status
	 - NOT switch any line status
	 - NOT switch anything in the topology
	 - NOT force any particular bus configuration
2025-03-05 15:02:45,414 - execution_func - INFO - 成功进行“发电机重调度”操作。

测试函数: obs_target_dispatch_impl
2025-03-05 15:02:45,415 - execution_func - INFO - 开始获取所有发电机的目标调度值，env_id: test_env
2025-03-05 15:02:45,416 - execution_func - INFO - 成功获取所有发电机的目标调度值。
状态: success
消息: 成功获取所有发电机的目标调度值。
数据 (前5项): [5.0, 0.0, 0.0, 0.0, 0.0]
数据长度: 6


(True,
 {'status': 'success',
  'message': '成功获取所有发电机的目标调度值。',
  'data': [5.0, 0.0, 0.0, 0.0, 0.0, 0.0]})

In [8]:
def generate_test_report(results):
    """生成测试报告"""
    from datetime import datetime
    
    report = ["# Grid2Op Observation函数测试报告"]
    report.append(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # 总体统计
    success_count = sum(1 for result in results.values() if result)
    total_count = len(results)
    report.append(f"## 测试总结")
    report.append(f"- 测试的函数总数: {total_count}")
    report.append(f"- 成功测试数: {success_count}")
    report.append(f"- 失败测试数: {total_count - success_count}")
    report.append(f"- 成功率: {success_count/total_count*100:.2f}%\n")
    
    # 按类别统计
    categories = {
        "发电机相关": "_gen_",
        "负载相关": "_load_",
        "线路相关": ["_p_or", "_q_or", "_v_or", "_a_or", "_theta_or", 
                 "_p_ex", "_q_ex", "_v_ex", "_a_ex", "_theta_ex",
                 "_rho", "_line_status", "_timestep_overflow", 
                 "_time_before_cooldown_line", "_time_next_maintenance",
                 "_duration_next_maintenance"],
        "变电站相关": "_time_before_cooldown_sub",
        "储能设备相关": "_storage_",
        "系统整体": ["_date_time", "_topo_vect", "_max_step", "_current_step", "_delta_time"],
        "告警系统": ["_alarm", "_alert", "_attack", "_attention_budget"]
    }
    
    report.append("## 按类别统计")
    for category, patterns in categories.items():
        if isinstance(patterns, str):
            category_results = {k: v for k, v in results.items() if patterns in k}
        else:
            category_results = {}
            for pattern in patterns:
                category_results.update({k: v for k, v in results.items() if pattern in k})
        
        category_success = sum(1 for result in category_results.values() if result)
        category_total = len(category_results)
        
        if category_total > 0:
            report.append(f"### {category}")
            report.append(f"- 测试数: {category_total}")
            report.append(f"- 成功数: {category_success}")
            report.append(f"- 成功率: {category_success/category_total*100:.2f}%")
            
            if category_success < category_total:
                report.append("- 失败的函数:")
                for name, success in category_results.items():
                    if not success:
                        report.append(f"  - {name}")
            report.append("")
    
    # 输出报告
    report_text = "\n".join(report)
    print(report_text)
    
    # 保存到文件
    with open("grid2op_observation_test_report.md", "w") as f:
        f.write(report_text)
    
    return report_text

# 生成报告
report = generate_test_report(test_results)

# Grid2Op Observation函数测试报告
生成时间: 2025-03-05 15:02:45

## 测试总结
- 测试的函数总数: 85
- 成功测试数: 85
- 失败测试数: 0
- 成功率: 100.00%

## 按类别统计
### 发电机相关
- 测试数: 14
- 成功数: 14
- 成功率: 100.00%

### 负载相关
- 测试数: 8
- 成功数: 8
- 成功率: 100.00%

### 线路相关
- 测试数: 32
- 成功数: 32
- 成功率: 100.00%

### 变电站相关
- 测试数: 2
- 成功数: 2
- 成功率: 100.00%

### 储能设备相关
- 测试数: 4
- 成功数: 4
- 成功率: 100.00%

### 系统整体
- 测试数: 5
- 成功数: 5
- 成功率: 100.00%

### 告警系统
- 测试数: 10
- 成功数: 10
- 成功率: 100.00%



In [9]:
# def debug_failed_function(func, env_manager, env_id, **kwargs):
#     """详细调试一个失败的函数"""
#     import inspect
    
#     print(f"\n调试函数: {func.__name__}")
    
#     # 检查函数的实现
#     print("\n函数的实现:")
#     print(inspect.getsource(func))
    
#     # 检查观察对象是否有必要的属性
#     obs = env_manager.get_latest_obs(env_id)
#     func_str = inspect.getsource(func)
    
#     # 尝试提取函数中使用的观察属性
#     # 这是一个简单的提取方法，可能需要根据实际情况优化
#     potential_attrs = []
#     for line in func_str.split('\n'):
#         if 'obs.' in line:
#             attr = line.split('obs.')[1].split('[')[0].split('.')[0].strip()
#             potential_attrs.append(attr)
    
#     print("\n检查观察对象的属性:")
#     for attr in set(potential_attrs):
#         if hasattr(obs, attr):
#             attr_value = getattr(obs, attr)
#             value_type = type(attr_value).__name__
#             value_size = len(attr_value) if hasattr(attr_value, '__len__') else '-'
#             print(f"- {attr}: 存在 (类型: {value_type}, 大小: {value_size})")
#         else:
#             print(f"- {attr}: 不存在")
    
#     # 尝试使用简化的方式调用函数
#     print("\n尝试直接访问观察对象属性:")
#     for attr in set(potential_attrs):
#         if hasattr(obs, attr):
#             try:
#                 attr_value = getattr(obs, attr)
#                 if isinstance(attr_value, np.ndarray) and attr_value.size > 0:
#                     print(f"- {attr}[0] = {attr_value[0]}")
#             except Exception as e:
#                 print(f"- 访问 {attr} 时出错: {str(e)}")
    
#     # 尝试直接执行函数的核心逻辑
#     print("\n尝试执行函数的核心逻辑:")
#     try:
#         if "gen_id" in kwargs:
#             gen_id = kwargs["gen_id"]
#             attr = potential_attrs[0] if potential_attrs else "Unknown"
#             if hasattr(obs, attr):
#                 attr_value = getattr(obs, attr)
#                 if gen_id is None:
#                     result = attr_value.tolist() if hasattr(attr_value, 'tolist') else attr_value
#                     print(f"结果 (前5项): {result[:5] if isinstance(result, list) and len(result) > 5 else result}")
#                 else:
#                     result = attr_value[gen_id].tolist() if hasattr(attr_value[gen_id], 'tolist') else attr_value[gen_id]
#                     print(f"结果: {result}")
#             else:
#                 print(f"属性 {attr} 不存在")
#         else:
#             print("无法直接执行，缺少必要参数")
#     except Exception as e:
#         print(f"执行时出错: {str(e)}")

# # 调试示例
# failed_funcs = [name for name, success in test_results.items() if not success]
# if failed_funcs:
#     print(f"\n开始调试失败的函数 (示例第一个): {failed_funcs[0]}")
#     # 根据函数名找到对应的函数对象
#     func_name = failed_funcs[0].split('_')[:-1]  # 移除 "_all" 或 "_specific" 后缀
#     real_func_name = '_'.join(func_name)
    
#     # 假设我们可以这样获取函数对象
#     import sys
#     this_module = sys.modules[__name__]
#     if hasattr(this_module, real_func_name):
#         func = getattr(this_module, real_func_name)
#         debug_kwargs = {}
#         if "specific" in failed_funcs[0]:
#             if "gen_" in real_func_name:
#                 debug_kwargs = {"gen_id": [0]}
#             elif "load_" in real_func_name:
#                 debug_kwargs = {"load_id": [0]}
#             elif "line_" in real_func_name or "_or_" in real_func_name or "_ex_" in real_func_name:
#                 debug_kwargs = {"line_id": [0]}
#             elif "storage_" in real_func_name:
#                 debug_kwargs = {"storage_id": [0]}
        
#         debug_failed_function(func, env_manager, env_id, **debug_kwargs)